# STAC Clip — STAC Catalog in, STAC Catalog out

In [ ]:
# Papermill parameters cell -- values here are overridden at execution time.
input_catalog = "input"  # path to the STAC Catalog directory
asset_name = "B04"
bbox = "-122.55 37.70 -122.35 37.85"  # MINX MINY MAXX MAXY, EPSG:4326
output_file = "clipped.tif"

In [ ]:
import os

import pystac
import rasterio
import rasterio.shutil
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from rio_stac.stac import create_stac_item

OUTPUT_DIR = "output"

## Read the staged STAC Catalog

Open the catalog from the input directory, take its first STAC Item, and resolve the requested
asset's local href. Because stage-in localized the data, the asset href points to a file on disk.

In [ ]:
def read_input_item(input_catalog, asset_name):
    """Load the staged catalog and return (item, local asset href) for ``asset_name``."""
    catalog_path = input_catalog
    if os.path.isdir(catalog_path):
        catalog_path = os.path.join(catalog_path, "catalog.json")
    catalog = pystac.Catalog.from_file(catalog_path)
    items = list(catalog.get_items(recursive=True))
    if not items:
        raise LookupError(f"No STAC Items found in catalog {catalog_path!r}.")
    item = items[0]
    if asset_name not in item.assets:
        raise KeyError(
            f"Asset {asset_name!r} not found. Available: {sorted(item.assets)}"
        )
    href = item.assets[asset_name].get_absolute_href() or item.assets[asset_name].href
    print(f"Input item {item.id!r}; asset {asset_name!r} -> {href}")
    return item, href

## Clip the raster to the bounding box

The bbox (EPSG:4326) is reprojected to the raster's CRS and used to window-read just that region,
which is then written out as a valid Cloud-Optimized GeoTIFF.

In [ ]:
def clip_asset(href, bbox, output_path):
    """Window-read ``href`` to ``bbox`` (lon/lat) and write it as a COG."""
    with rasterio.open(href) as src:
        # The bbox is in EPSG:4326; reproject it to the raster's CRS.
        dst_bounds = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(*dst_bounds, transform=src.transform)
        window = window.round_offsets().round_lengths()

        if window.width <= 0 or window.height <= 0:
            raise ValueError(
                f"bbox {bbox} does not intersect the raster bounds {src.bounds}."
            )

        data = src.read(window=window)
        transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=data.shape[1],
            width=data.shape[2],
            transform=transform,
            tiled=False,
        )
        # Drop any source block sizes that no longer fit the clipped raster.
        for key in ("blockxsize", "blockysize"):
            profile.pop(key, None)

        # Write to an in-memory GeoTIFF, then CreateCopy it to a valid COG.
        with MemoryFile() as mem:
            with mem.open(**profile) as tmp:
                tmp.write(data)
                rasterio.shutil.copy(
                    tmp, output_path, driver="COG", overwrite=True
                )
    print(f"Wrote clipped COG to {output_path}")
    return output_path

## Write the output STAC Catalog (stage-out)

Build a STAC Item for the clipped COG with `rio-stac` (including projection and raster extension
metadata), link it back to the input Item for provenance (`derived_from`), and save a self-contained
STAC Catalog into `output/` — `catalog.json`, the Item JSON, and the COG with relative hrefs.

In [ ]:
def write_stac_catalog(clipped_path, output_file, src_item, asset_name, output_dir=OUTPUT_DIR):
    """Emit a self-contained STAC Catalog describing the clipped COG."""
    out_id = f"{src_item.id}-clip"
    item = create_stac_item(
        source=clipped_path,
        input_datetime=src_item.datetime,
        id=out_id,
        asset_name=asset_name,
        asset_href=output_file,  # relative to the item JSON (sibling in output/)
        asset_media_type=pystac.MediaType.COG,
        asset_roles=["data"],
        with_proj=True,
        with_raster=True,
    )

    # Provenance: link the output Item back to the input Item.
    src_href = src_item.get_self_href()
    if src_href:
        item.add_link(
            pystac.Link(rel="derived_from", target=src_href, media_type=pystac.MediaType.JSON)
        )

    catalog = pystac.Catalog(
        id="stac-clip-output",
        description="Clipped raster output produced by the stac-clip application.",
    )
    catalog.add_item(item)

    # Flat, self-contained layout: catalog.json, <out_id>.json and the COG all in output/.
    catalog.set_self_href(os.path.join(output_dir, "catalog.json"))
    item.set_self_href(os.path.join(output_dir, f"{out_id}.json"))
    catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)

    print(f"Wrote STAC Catalog to {os.path.join(output_dir, 'catalog.json')}")
    return catalog

## Run the workflow

In [ ]:
bbox_values = [float(v) for v in bbox.split()]
if len(bbox_values) != 4:
    raise ValueError("bbox must have four values: 'MINX MINY MAXX MAXY'")

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, output_file)

src_item, href = read_input_item(input_catalog, asset_name)
clip_asset(href, bbox_values, output_path)
write_stac_catalog(output_path, output_file, src_item, asset_name)

print(f"\nDone. Output STAC Catalog in {OUTPUT_DIR}/")